# Paper Reproduction

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, GridSearchCV , train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from metric_learn import MLKR
from sklearn.ensemble import RandomForestRegressor

# REu1

In [3]:
# read the CSV file into a DataFrame
df = pd.read_csv("/home/sk/Projects/Astrophysics/data/ATLAS.csv")
print("Original dataset shape:", df.shape)

# extratcting the features and target variable
features = ["Sp2", "flux_ap2_36", "flux_ap2_45", "flux_ap2_58", "flux_ap2_80", "MAG_APER_4_G", "MAG_APER_4_R", "MAG_APER_4_I", "MAG_APER_4_Z"]
target = "z"

# converting the features and target variable to numpy arrays
X = df[features].to_numpy(dtype=np.float32)
y = df[target].to_numpy(dtype=np.float32)

# splitting the dataset into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

np.random.seed(42)

test_indices = np.random.choice(
    len(X),
    round(len(X) * 0.3),
    replace=False
)

train_indices = np.array(
    list(set(range(len(X))) - set(test_indices))
)

X_train = X[train_indices]
X_test = X[test_indices]
y_train = y[train_indices]
y_test = y[test_indices]

print("DATA PARTITION")
print("Total sources   :", len(X))
print("Training sources:", len(X_train))
print("Testing sources :", len(X_test))


def eta_015(y_true, y_pred):
    """
    Catastrophic outlier fraction.
    A source is an outlier when: |y_pred - y_true| > 0.15 * (1 + y_true)
    """
    
    error = np.abs(y_pred - y_true)
    outliers = (error > 0.15 * (1 + y_true))

    return np.mean(outliers)

# creating a custom scorer for eta_015
eta_015_scorer = make_scorer(eta_015, greater_is_better=False) # Therefore GridSearchCV will choose the k with the LOWEST eta_0.15.

# defining the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(metric="euclidean"))
])

param_grid = {
    "knn__n_neighbors": range(2, 20)
}

# defining the K-Fold cross-validation strategy
cv = KFold(n_splits=10, shuffle=True, random_state=10)

# defining the GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=eta_015_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)
# fitting the GridSearchCV
grid_search.fit(X_train,y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["knn__n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:",best_cv_eta)

# GridSearchCV refit the best model on the whole training set, so we can use it to predict on the test set
y_pred = best_model.predict(X_test)

# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15 on the test set
eta_015_value = eta_015(y_test, y_pred) * 100

# eta_2sigma 
sigma = np.std(delta_z)
eta_2sigma_value = (np.mean(np.abs(delta_z) > 2 * sigma)* 100)
sigma_value = np.std(delta_z)

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = (1.4826 * mad)

# R^2 score
r2 = r2_score(y_test, y_pred)

# Mean squared error
mse = mean_squared_error(y_test, y_pred)

# Root mean squared error
rmse = np.sqrt(mse)

# Mean absolute error
mae = mean_absolute_error(y_test,y_pred)

# printing the results
print(f"Best k: {best_k}")
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Original dataset shape: (1311, 122)
DATA PARTITION
Total sources   : 1311
Training sources: 918
Testing sources : 393
Best k: 5
Best CV eta_0.15: 0.08932154801720019
Best k: 5
eta_0.15: 8.14%
eta_2sigma: 6.62%
sigma: 0.1022
sigma_NMAD: 0.0523
R²: 0.6479
MSE: 0.0597
RMSE: 0.2442
MAE: 0.1072


# REu2

In [4]:
# read the CSV file into a DataFrame
df_elais = pd.read_csv("/home/sk/Projects/Astrophysics/data/ELAIS-S1.csv")  # training dataset
df_cdfs = pd.read_csv("/home/sk/Projects/Astrophysics/data/CDFS.csv")  # testing dataset
print("Original ELAIS-S1 dataset shape:", df_elais.shape)
print("Original CDFS dataset shape:", df_cdfs.shape)

# extratcting the features and target variable
features = ["Sp2", "flux_ap2_36", "flux_ap2_45", "flux_ap2_58", "flux_ap2_80", "MAG_APER_4_G", "MAG_APER_4_R", "MAG_APER_4_I", "MAG_APER_4_Z"]
target = "z"

# converting the features and target variable to numpy arrays
X_train = df_elais[features].to_numpy(dtype=np.float32)
y_train = df_elais[target].to_numpy(dtype=np.float32)
X_test = df_cdfs[features].to_numpy(dtype=np.float32)
y_test = df_cdfs[target].to_numpy(dtype=np.float32)


def eta_015(y_true, y_pred):
    """
    Catastrophic outlier fraction.
    A source is an outlier when: |y_pred - y_true| > 0.15 * (1 + y_true)
    """
    
    error = np.abs(y_pred - y_true)
    outliers = (error > 0.15 * (1 + y_true))

    return np.mean(outliers)

# creating a custom scorer for eta_015
eta_015_scorer = make_scorer(eta_015, greater_is_better=False) # Therefore GridSearchCV will choose the k with the LOWEST eta_0.15.

# defining the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(metric="euclidean"))
])

param_grid = {
    "knn__n_neighbors": range(2, 20)
}

# defining the K-Fold cross-validation strategy
cv = KFold(n_splits=10, shuffle=True, random_state=10)

# defining the GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=eta_015_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)
# fitting the GridSearchCV
grid_search.fit(X_train,y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["knn__n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:",best_cv_eta)

# GridSearchCV refit the best model on the whole training set, so we can use it to predict on the test set
y_pred = best_model.predict(X_test)

# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15 on the test set
eta_015_value = eta_015(y_test, y_pred) * 100

# eta_2sigma 
sigma = np.std(delta_z)
eta_2sigma_value = (np.mean(np.abs(delta_z) > 2 * sigma)* 100)
sigma_value = np.std(delta_z)

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = (1.4826 * mad)

# R^2 score
r2 = r2_score(y_test, y_pred)

# Mean squared error
mse = mean_squared_error(y_test, y_pred)

# Root mean squared error
rmse = np.sqrt(mse)

# Mean absolute error
mae = mean_absolute_error(y_test,y_pred)

# printing the results
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Original ELAIS-S1 dataset shape: (495, 122)
Original CDFS dataset shape: (816, 122)
Best k: 5
Best CV eta_0.15: 0.10710204081632653
eta_0.15: 8.70%
eta_2sigma: 5.51%
sigma: 0.1196
sigma_NMAD: 0.0618
R²: 0.6008
MSE: 0.0650
RMSE: 0.2549
MAE: 0.1172


# REu3

In [5]:
# read the CSV file into a DataFrame
df_elais = pd.read_csv("/home/sk/Projects/Astrophysics/data/ELAIS-S1.csv")  # training dataset
df_cdfs = pd.read_csv("/home/sk/Projects/Astrophysics/data/CDFS.csv")  # testing dataset
print("Original ELAIS-S1 dataset shape:", df_elais.shape)
print("Original CDFS dataset shape:", df_cdfs.shape)

# extratcting the features and target variable
features = ["Sp2", "flux_ap2_36", "flux_ap2_45", "flux_ap2_58", "flux_ap2_80", "MAG_APER_4_G", "MAG_APER_4_R", "MAG_APER_4_I", "MAG_APER_4_Z"]
target = "z"

# converting the features and target variable to numpy arrays
X_train = df_cdfs[features].to_numpy(dtype=np.float32)
y_train = df_cdfs[target].to_numpy(dtype=np.float32)
X_test = df_elais[features].to_numpy(dtype=np.float32)
y_test = df_elais[target].to_numpy(dtype=np.float32)


def eta_015(y_true, y_pred):
    """
    Catastrophic outlier fraction.
    A source is an outlier when: |y_pred - y_true| > 0.15 * (1 + y_true)
    """
    
    error = np.abs(y_pred - y_true)
    outliers = (error > 0.15 * (1 + y_true))

    return np.mean(outliers)

# creating a custom scorer for eta_015
eta_015_scorer = make_scorer(eta_015, greater_is_better=False) # Therefore GridSearchCV will choose the k with the LOWEST eta_0.15.

# defining the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(metric="euclidean"))
])

param_grid = {
    "knn__n_neighbors": range(2, 20)
}

# defining the K-Fold cross-validation strategy
cv = KFold(n_splits=10, shuffle=True, random_state=10)

# defining the GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=eta_015_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)
# fitting the GridSearchCV
grid_search.fit(X_train,y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["knn__n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:",best_cv_eta)

# GridSearchCV refit the best model on the whole training set, so we can use it to predict on the test set
y_pred = best_model.predict(X_test)

# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15 on the test set
eta_015_value = eta_015(y_test, y_pred) * 100

# eta_2sigma 
sigma = np.std(delta_z)
eta_2sigma_value = (np.mean(np.abs(delta_z) > 2 * sigma)* 100)
sigma_value = np.std(delta_z)

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = (1.4826 * mad)

# R^2 score
r2 = r2_score(y_test, y_pred)

# Mean squared error
mse = mean_squared_error(y_test, y_pred)

# Root mean squared error
rmse = np.sqrt(mse)

# Mean absolute error
mae = mean_absolute_error(y_test,y_pred)

# printing the results
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Original ELAIS-S1 dataset shape: (495, 122)
Original CDFS dataset shape: (816, 122)
Best k: 11
Best CV eta_0.15: 0.08706714844926226
eta_0.15: 8.28%
eta_2sigma: 4.85%
sigma: 0.1233
sigma_NMAD: 0.0598
R²: 0.5262
MSE: 0.0889
RMSE: 0.2981
MAE: 0.1251


# RMa1

In [6]:
# read the CSV file into a DataFrame
df = pd.read_csv("/home/sk/Projects/Astrophysics/data/ATLAS.csv")
print("Original dataset shape:", df.shape)

# extratcting the features and target variable
features = ["Sp2", "flux_ap2_36", "flux_ap2_45", "flux_ap2_58", "flux_ap2_80", "MAG_APER_4_G", "MAG_APER_4_R", "MAG_APER_4_I", "MAG_APER_4_Z"]
target = "z"

# converting the features and target variable to numpy arrays
X = df[features].to_numpy(dtype=np.float32)
y = df[target].to_numpy(dtype=np.float32)

# splitting the dataset into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

np.random.seed(42)

test_indices = np.random.choice(
    len(X),
    round(len(X) * 0.3),
    replace=False
)

train_indices = np.array(
    list(set(range(len(X))) - set(test_indices))
)

X_train = X[train_indices]
X_test = X[test_indices]
y_train = y[train_indices]
y_test = y[test_indices]

print("DATA PARTITION")
print("Total sources   :", len(X))
print("Training sources:", len(X_train))
print("Testing sources :", len(X_test))


def eta_015(y_true, y_pred):
    """
    Catastrophic outlier fraction.
    A source is an outlier when: |y_pred - y_true| > 0.15 * (1 + y_true)
    """
    
    error = np.abs(y_pred - y_true)
    outliers = (error > 0.15 * (1 + y_true))

    return np.mean(outliers)

# creating a custom scorer for eta_015
eta_015_scorer = make_scorer(eta_015, greater_is_better=False) # Therefore GridSearchCV will choose the k with the LOWEST eta_0.15.


class MahalanobisKNN(BaseEstimator, RegressorMixin):
    def __init__(self, n_neighbors=2):
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        self.covariance_ = np.cov(X, rowvar=False)
        self.VI_ = np.linalg.pinv(self.covariance_)

        self.knn_ = KNeighborsRegressor(
            n_neighbors=self.n_neighbors,
            metric="mahalanobis",
            metric_params={"VI": self.VI_}
        )

        self.knn_.fit(X, y)
        return self

    def predict(self, X):
        return self.knn_.predict(X)



# defining the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', MahalanobisKNN())
])

param_grid = {
    "knn__n_neighbors": range(2, 20)
}

# defining the K-Fold cross-validation strategy
cv = KFold(n_splits=10, shuffle=True, random_state=10)

# defining the GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=eta_015_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)
# fitting the GridSearchCV
grid_search.fit(X_train,y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["knn__n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:",best_cv_eta)

# GridSearchCV refit the best model on the whole training set, so we can use it to predict on the test set
y_pred = best_model.predict(X_test)

# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15 on the test set
eta_015_value = eta_015(y_test, y_pred) * 100

# eta_2sigma 
sigma = np.std(delta_z)
eta_2sigma_value = (np.mean(np.abs(delta_z) > 2 * sigma)* 100)
sigma_value = np.std(delta_z)

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = (1.4826 * mad)

# R^2 score
r2 = r2_score(y_test, y_pred)

# Mean squared error
mse = mean_squared_error(y_test, y_pred)

# Root mean squared error
rmse = np.sqrt(mse)

# Mean absolute error
mae = mean_absolute_error(y_test,y_pred)

# printing the results
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Original dataset shape: (1311, 122)
DATA PARTITION
Total sources   : 1311
Training sources: 918
Testing sources : 393
Best k: 5
Best CV eta_0.15: 0.07953893932154801
eta_0.15: 5.85%
eta_2sigma: 5.09%
sigma: 0.1164
sigma_NMAD: 0.0312
R²: 0.5928
MSE: 0.0690
RMSE: 0.2626
MAE: 0.0891


# RMa2

In [7]:
# read the CSV file into a DataFrame
df_elais = pd.read_csv("/home/sk/Projects/Astrophysics/data/ELAIS-S1.csv")  # training dataset
df_cdfs = pd.read_csv("/home/sk/Projects/Astrophysics/data/CDFS.csv")  # testing dataset
print("Original ELAIS-S1 dataset shape:", df_elais.shape)
print("Original CDFS dataset shape:", df_cdfs.shape)

# extratcting the features and target variable
features = ["Sp2", "flux_ap2_36", "flux_ap2_45", "flux_ap2_58", "flux_ap2_80", "MAG_APER_4_G", "MAG_APER_4_R", "MAG_APER_4_I", "MAG_APER_4_Z"]
target = "z"

# converting the features and target variable to numpy arrays
X_train = df_elais[features].to_numpy(dtype=np.float32)
y_train = df_elais[target].to_numpy(dtype=np.float32)
X_test = df_cdfs[features].to_numpy(dtype=np.float32)
y_test = df_cdfs[target].to_numpy(dtype=np.float32)


def eta_015(y_true, y_pred):
    """
    Catastrophic outlier fraction.
    A source is an outlier when: |y_pred - y_true| > 0.15 * (1 + y_true)
    """
    
    error = np.abs(y_pred - y_true)
    outliers = (error > 0.15 * (1 + y_true))

    return np.mean(outliers)

# creating a custom scorer for eta_015
eta_015_scorer = make_scorer(eta_015, greater_is_better=False) # Therefore GridSearchCV will choose the k with the LOWEST eta_0.15.


class MahalanobisKNN(BaseEstimator, RegressorMixin):
    def __init__(self, n_neighbors=2):
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        self.covariance_ = np.cov(X, rowvar=False)
        self.VI_ = np.linalg.pinv(self.covariance_)

        self.knn_ = KNeighborsRegressor(
            n_neighbors=self.n_neighbors,
            metric="mahalanobis",
            metric_params={"VI": self.VI_}
        )

        self.knn_.fit(X, y)
        return self

    def predict(self, X):
        return self.knn_.predict(X)



# defining the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', MahalanobisKNN())
])

param_grid = {
    "knn__n_neighbors": range(2, 20)
}

# defining the K-Fold cross-validation strategy
cv = KFold(n_splits=10, shuffle=True, random_state=10)

# defining the GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=eta_015_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)
# fitting the GridSearchCV
grid_search.fit(X_train,y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["knn__n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:",best_cv_eta)

# GridSearchCV refit the best model on the whole training set, so we can use it to predict on the test set
y_pred = best_model.predict(X_test)

# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15 on the test set
eta_015_value = eta_015(y_test, y_pred) * 100

# eta_2sigma 
sigma = np.std(delta_z)
eta_2sigma_value = (np.mean(np.abs(delta_z) > 2 * sigma)* 100)
sigma_value = np.std(delta_z)

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = (1.4826 * mad)

# R^2 score
r2 = r2_score(y_test, y_pred)

# Mean squared error
mse = mean_squared_error(y_test, y_pred)

# Root mean squared error
rmse = np.sqrt(mse)

# Mean absolute error
mae = mean_absolute_error(y_test,y_pred)

# printing the results
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Original ELAIS-S1 dataset shape: (495, 122)
Original CDFS dataset shape: (816, 122)
Best k: 3
Best CV eta_0.15: 0.07069387755102043
eta_0.15: 8.70%
eta_2sigma: 5.39%
sigma: 0.1462
sigma_NMAD: 0.0363
R²: 0.5843
MSE: 0.0676
RMSE: 0.2601
MAE: 0.1021


# RMa3

In [8]:
# read the CSV file into a DataFrame
df_elais = pd.read_csv("/home/sk/Projects/Astrophysics/data/ELAIS-S1.csv")  # training dataset
df_cdfs = pd.read_csv("/home/sk/Projects/Astrophysics/data/CDFS.csv")  # testing dataset
print("Original ELAIS-S1 dataset shape:", df_elais.shape)
print("Original CDFS dataset shape:", df_cdfs.shape)

# extratcting the features and target variable
features = ["Sp2", "flux_ap2_36", "flux_ap2_45", "flux_ap2_58", "flux_ap2_80", "MAG_APER_4_G", "MAG_APER_4_R", "MAG_APER_4_I", "MAG_APER_4_Z"]
target = "z"

# converting the features and target variable to numpy arrays
X_train = df_cdfs[features].to_numpy(dtype=np.float32)
y_train = df_cdfs[target].to_numpy(dtype=np.float32)
X_test = df_elais[features].to_numpy(dtype=np.float32)
y_test = df_elais[target].to_numpy(dtype=np.float32)


def eta_015(y_true, y_pred):
    """
    Catastrophic outlier fraction.
    A source is an outlier when: |y_pred - y_true| > 0.15 * (1 + y_true)
    """
    
    error = np.abs(y_pred - y_true)
    outliers = (error > 0.15 * (1 + y_true))

    return np.mean(outliers)

# creating a custom scorer for eta_015
eta_015_scorer = make_scorer(eta_015, greater_is_better=False) # Therefore GridSearchCV will choose the k with the LOWEST eta_0.15.


class MahalanobisKNN(BaseEstimator, RegressorMixin):
    def __init__(self, n_neighbors=2):
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        self.covariance_ = np.cov(X, rowvar=False)
        self.VI_ = np.linalg.pinv(self.covariance_)

        self.knn_ = KNeighborsRegressor(
            n_neighbors=self.n_neighbors,
            metric="mahalanobis",
            metric_params={"VI": self.VI_}
        )

        self.knn_.fit(X, y)
        return self

    def predict(self, X):
        return self.knn_.predict(X)



# defining the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', MahalanobisKNN())
])

param_grid = {
    "knn__n_neighbors": range(2, 20)
}

# defining the K-Fold cross-validation strategy
cv = KFold(n_splits=10, shuffle=True, random_state=10)

# defining the GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=eta_015_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)
# fitting the GridSearchCV
grid_search.fit(X_train,y_train)


# best model and best k
best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["knn__n_neighbors"]
best_cv_eta = -grid_search.best_score_

print("Best k:", best_k)
print("Best CV eta_0.15:",best_cv_eta)

# GridSearchCV refit the best model on the whole training set, so we can use it to predict on the test set
y_pred = best_model.predict(X_test)

# Normalized residuals
delta_z = (y_test - y_pred) / (1 + y_test)

# eta_0.15 on the test set
eta_015_value = eta_015(y_test, y_pred) * 100

# eta_2sigma 
sigma = np.std(delta_z)
eta_2sigma_value = (np.mean(np.abs(delta_z) > 2 * sigma)* 100)
sigma_value = np.std(delta_z)

# sigma_NMAD
median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = (1.4826 * mad)

# R^2 score
r2 = r2_score(y_test, y_pred)

# Mean squared error
mse = mean_squared_error(y_test, y_pred)

# Root mean squared error
rmse = np.sqrt(mse)

# Mean absolute error
mae = mean_absolute_error(y_test,y_pred)

# printing the results
print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Original ELAIS-S1 dataset shape: (495, 122)
Original CDFS dataset shape: (816, 122)
Best k: 5
Best CV eta_0.15: 0.06991869918699187
eta_0.15: 7.68%
eta_2sigma: 5.45%
sigma: 0.1330
sigma_NMAD: 0.0358
R²: 0.5433
MSE: 0.0857
RMSE: 0.2927
MAE: 0.1079


# RML1 

In [9]:
# read the CSV file into a DataFrame
df = pd.read_csv("/home/sk/Projects/Astrophysics/data/ATLAS.csv")
print("Original dataset shape:", df.shape)

# extracting the features and target variable
features = [
    "Sp2",
    "flux_ap2_36",
    "flux_ap2_45",
    "flux_ap2_58",
    "flux_ap2_80",
    "MAG_APER_4_G",
    "MAG_APER_4_R",
    "MAG_APER_4_I",
    "MAG_APER_4_Z"
]

target = "z"

# converting the features and target variable to numpy arrays
X = df[features].to_numpy(dtype=np.float32)
y = df[target].to_numpy(dtype=np.float32)

# splitting the dataset into training and testing sets
np.random.seed(42)

test_indices = np.random.choice(
    len(X),
    round(len(X) * 0.3),
    replace=False
)

train_indices = np.array(
    list(set(range(len(X))) - set(test_indices))
)

X_train = X[train_indices]
X_test = X[test_indices]
y_train = y[train_indices]
y_test = y[test_indices]

print("DATA PARTITION")
print("Total sources   :", len(X))
print("Training sources:", len(X_train))
print("Testing sources :", len(X_test))


def eta_015(y_true, y_pred):
    error = np.abs(y_pred - y_true)
    outliers = error > 0.15 * (1 + y_true)
    return np.mean(outliers)


eta_015_scorer = make_scorer(
    eta_015,
    greater_is_better=False
)


def metricLearnRegression(X_train, y_train):
    model = MLKR(max_iter=100)
    model.fit(X_train, y_train.ravel())
    return model


def mlkr_knn_model(k, X_train, X_test, y_train):
    
    scaler = StandardScaler()

    X_train_norm = scaler.fit_transform(X_train)
    X_test_norm = scaler.transform(X_test)

    B = metricLearnRegression(
        X_train_norm,
        y_train
    )

    X_train_mlkr = B.transform(X_train_norm)
    X_test_mlkr = B.transform(X_test_norm)

    knn = KNeighborsRegressor(
        n_neighbors=k,
        metric="euclidean"
    )

    knn.fit(
        X_train_mlkr,
        y_train.ravel()
    )

    y_pred = knn.predict(X_test_mlkr)

    return y_pred


cv = KFold(
    n_splits=10,
    shuffle=True,
    random_state=10
)

k_values = range(2, 20)

cv_eta = []

for k in k_values:

    eta_scores = []

    for train_idx, val_idx in cv.split(X_train):

        X_train_cross = X_train[train_idx]
        X_test_cross = X_train[val_idx]

        y_train_cross = y_train[train_idx]
        y_test_cross = y_train[val_idx]

        y_pred_cross = mlkr_knn_model(
            k,
            X_train_cross,
            X_test_cross,
            y_train_cross
        )

        eta = eta_015(
            y_test_cross,
            y_pred_cross
        )

        eta_scores.append(eta)

    mean_eta = np.mean(eta_scores)

    cv_eta.append(mean_eta)

    print(
        f"k = {k:2d} | "
        f"CV eta_0.15 = {mean_eta:.6f}"
    )


best_k_index = np.argmin(
    np.array(cv_eta)
)

best_k = list(k_values)[best_k_index]

best_cv_eta = cv_eta[best_k_index]

print("\nBest k:", best_k)
print("Best CV eta_0.15:", best_cv_eta)


# final model

scaler = StandardScaler()

X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

B = metricLearnRegression(
    X_train_norm,
    y_train
)

X_train_mlkr = B.transform(
    X_train_norm
)

X_test_mlkr = B.transform(
    X_test_norm
)

knn = KNeighborsRegressor(
    n_neighbors=best_k,
    metric="euclidean"
)

knn.fit(
    X_train_mlkr,
    y_train.ravel()
)

y_pred = knn.predict(
    X_test_mlkr
)


# normalized residuals

delta_z = (
    y_test - y_pred
) / (
    1 + y_test
)


# eta_0.15

eta_015_value = (
    eta_015(
        y_test,
        y_pred
    ) * 100
)


# eta_2sigma

sigma = np.std(delta_z)

eta_2sigma_value = (
    np.mean(
        np.abs(delta_z) > 2 * sigma
    ) * 100
)

sigma_value = np.std(delta_z)


# sigma_NMAD

median_delta_z = np.median(
    delta_z
)

mad = np.median(
    np.abs(
        delta_z - median_delta_z
    )
)

sigma_nmad = 1.4826 * mad


# R²

r2 = r2_score(
    y_test,
    y_pred
)


# MSE

mse = mean_squared_error(
    y_test,
    y_pred
)


# RMSE

rmse = np.sqrt(mse)


# MAE

mae = mean_absolute_error(
    y_test,
    y_pred
)


# printing the results

print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Original dataset shape: (1311, 122)
DATA PARTITION
Total sources   : 1311
Training sources: 918
Testing sources : 393


/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: '

k =  2 | CV eta_0.15 = 0.083851


/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: '

k =  3 | CV eta_0.15 = 0.087136


...

/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: '

k = 18 | CV eta_0.15 = 0.093681


/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: '

k = 19 | CV eta_0.15 = 0.095855

Best k: 4
Best CV eta_0.15: 0.0817009077878643
eta_0.15: 6.87%
eta_2sigma: 4.83%
sigma: 0.1032
sigma_NMAD: 0.0450
R²: 0.7259
MSE: 0.0464
RMSE: 0.2155
MAE: 0.0935


/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


# RMl2

In [16]:
# read the CSV file into a DataFrame
df_elais = pd.read_csv("/home/sk/Projects/Astrophysics/data/ELAIS-S1.csv")  # training dataset
df_cdfs = pd.read_csv("/home/sk/Projects/Astrophysics/data/CDFS.csv")  # testing dataset
print("Original ELAIS-S1 dataset shape:", df_elais.shape)
print("Original CDFS dataset shape:", df_cdfs.shape)

# extratcting the features and target variable
features = ["Sp2", "flux_ap2_36", "flux_ap2_45", "flux_ap2_58", "flux_ap2_80", "MAG_APER_4_G", "MAG_APER_4_R", "MAG_APER_4_I", "MAG_APER_4_Z"]
target = "z"

# converting the features and target variable to numpy arrays
X_train = df_cdfs[features].to_numpy(dtype=np.float32)
y_train = df_cdfs[target].to_numpy(dtype=np.float32)
X_test = df_elais[features].to_numpy(dtype=np.float32)
y_test = df_elais[target].to_numpy(dtype=np.float32)


def eta_015(y_true, y_pred):
    error = np.abs(y_pred - y_true)
    outliers = error > 0.15 * (1 + y_true)
    return np.mean(outliers)


eta_015_scorer = make_scorer(
    eta_015,
    greater_is_better=False
)


def metricLearnRegression(X_train, y_train):
    model = MLKR(max_iter=100)
    model.fit(X_train, y_train.ravel())
    return model


def mlkr_knn_model(k, X_train, X_test, y_train):
    
    scaler = StandardScaler()

    X_train_norm = scaler.fit_transform(X_train)
    X_test_norm = scaler.transform(X_test)

    B = metricLearnRegression(
        X_train_norm,
        y_train
    )

    X_train_mlkr = B.transform(X_train_norm)
    X_test_mlkr = B.transform(X_test_norm)

    knn = KNeighborsRegressor(
        n_neighbors=k,
        metric="euclidean"
    )

    knn.fit(
        X_train_mlkr,
        y_train.ravel()
    )

    y_pred = knn.predict(X_test_mlkr)

    return y_pred


cv = KFold(
    n_splits=10,
    shuffle=True,
    random_state=10
)

k_values = range(2, 20)

cv_eta = []

for k in k_values:

    eta_scores = []

    for train_idx, val_idx in cv.split(X_train):

        X_train_cross = X_train[train_idx]
        X_test_cross = X_train[val_idx]

        y_train_cross = y_train[train_idx]
        y_test_cross = y_train[val_idx]

        y_pred_cross = mlkr_knn_model(
            k,
            X_train_cross,
            X_test_cross,
            y_train_cross
        )

        eta = eta_015(
            y_test_cross,
            y_pred_cross
        )

        eta_scores.append(eta)

    mean_eta = np.mean(eta_scores)

    cv_eta.append(mean_eta)

    print(
        f"k = {k:2d} | "
        f"CV eta_0.15 = {mean_eta:.6f}"
    )


best_k_index = np.argmin(
    np.array(cv_eta)
)

best_k = list(k_values)[best_k_index]

best_cv_eta = cv_eta[best_k_index]

print("\nBest k:", best_k)
print("Best CV eta_0.15:", best_cv_eta)


# final model

scaler = StandardScaler()

X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

B = metricLearnRegression(
    X_train_norm,
    y_train
)

X_train_mlkr = B.transform(
    X_train_norm
)

X_test_mlkr = B.transform(
    X_test_norm
)

knn = KNeighborsRegressor(
    n_neighbors=best_k,
    metric="euclidean"
)

knn.fit(
    X_train_mlkr,
    y_train.ravel()
)

y_pred = knn.predict(
    X_test_mlkr
)


# normalized residuals

delta_z = (
    y_test - y_pred
) / (
    1 + y_test
)


# eta_0.15

eta_015_value = (
    eta_015(
        y_test,
        y_pred
    ) * 100
)


# eta_2sigma

sigma = np.std(delta_z)

eta_2sigma_value = (
    np.mean(
        np.abs(delta_z) > 2 * sigma
    ) * 100
)

sigma_value = np.std(delta_z)


# sigma_NMAD

median_delta_z = np.median(
    delta_z
)

mad = np.median(
    np.abs(
        delta_z - median_delta_z
    )
)

sigma_nmad = 1.4826 * mad


# R²

r2 = r2_score(
    y_test,
    y_pred
)


# MSE

mse = mean_squared_error(
    y_test,
    y_pred
)


# RMSE

rmse = np.sqrt(mse)


# MAE

mae = mean_absolute_error(
    y_test,
    y_pred
)


# printing the results

print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma_value:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Original ELAIS-S1 dataset shape: (495, 122)
Original CDFS dataset shape: (816, 122)


/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: '

k =  2 | CV eta_0.15 = 0.076031


/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: '

k =  3 | CV eta_0.15 = 0.072328


...

/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: '

k = 18 | CV eta_0.15 = 0.079690


/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: '

k = 19 | CV eta_0.15 = 0.082129

Best k: 8
Best CV eta_0.15: 0.06988858777476663
eta_0.15: 8.69%
eta_2sigma: 3.64%
sigma: 0.1280
sigma_NMAD: 0.0591
R²: 0.5862
MSE: 0.0776
RMSE: 0.2786
MAE: 0.1191


/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/sk/Projects/Astrophysics/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


# RRF1

In [21]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, make_scorer

df_elais = pd.read_csv("/home/sk/Projects/Astrophysics/data/ELAIS-S1.csv")
df_cdfs = pd.read_csv("/home/sk/Projects/Astrophysics/data/CDFS.csv")

print("Original ELAIS-S1 dataset shape:", df_elais.shape)
print("Original CDFS dataset shape:", df_cdfs.shape)

features = ["Sp2", "flux_ap2_36", "flux_ap2_45", "flux_ap2_58", "flux_ap2_80",
            "MAG_APER_4_G", "MAG_APER_4_R", "MAG_APER_4_I", "MAG_APER_4_Z"]
target = "z"

X_train = df_elais[features].to_numpy(dtype=np.float32)
y_train = df_elais[target].to_numpy(dtype=np.float32)
X_test = df_cdfs[features].to_numpy(dtype=np.float32)
y_test = df_cdfs[target].to_numpy(dtype=np.float32)


def eta_015(y_true, y_pred):
    error = np.abs(y_pred - y_true)
    outliers = error > 0.15 * (1 + y_true)
    return np.mean(outliers)


eta_015_scorer = make_scorer(eta_015, greater_is_better=False)

pipeline = Pipeline([
    ("rf", RandomForestRegressor(random_state=42, n_jobs=-1))
])

param_grid = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [10, 20, 30, None],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4],
    "rf__max_features": [1.0, "sqrt", "log2"]
}

cv = KFold(n_splits=10, shuffle=True, random_state=10)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=eta_015_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_cv_eta = -grid_search.best_score_

print("Best parameters:", best_params)
print("Best CV eta_0.15:", best_cv_eta)

y_pred = best_model.predict(X_test)

delta_z = (y_test - y_pred) / (1 + y_test)

eta_015_value = eta_015(y_test, y_pred) * 100

sigma = np.std(delta_z)
eta_2sigma_value = np.mean(np.abs(delta_z) > 2 * sigma) * 100

median_delta_z = np.median(delta_z)
mad = np.median(np.abs(delta_z - median_delta_z))
sigma_nmad = 1.4826 * mad

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print(f"eta_0.15: {eta_015_value:.2f}%")
print(f"eta_2sigma: {eta_2sigma_value:.2f}%")
print(f"sigma: {sigma:.4f}")
print(f"sigma_NMAD: {sigma_nmad:.4f}")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

import numpy as np
import matplotlib.pyplot as plt

def plot_photoz_results(y_true, y_pred, title="Random Forest"):
    delta_z = (y_true - y_pred) / (1 + y_true)

    sigma = np.std(delta_z)
    sigma_nmad = 1.4826 * np.median(np.abs(delta_z - np.median(delta_z)))
    eta = np.mean(np.abs(delta_z) > 0.15) * 100

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5, 7), sharex=True)

    zmax = max(np.max(y_true), np.max(y_pred))
    z = np.linspace(0, zmax, 500)

    ax1.scatter(y_true, y_pred, s=5, c="black", alpha=0.7)
    ax1.plot(z, z, "r--", linewidth=1)
    ax1.plot(z, z + 0.15 * (1 + z), "b--", linewidth=1)
    ax1.plot(z, z - 0.15 * (1 + z), "b--", linewidth=1)

    ax1.set_xlim(0, zmax)
    ax1.set_ylim(0, zmax)
    ax1.set_ylabel(r"$z_{\rm photo}$")
    ax1.set_title(title)

    ax1.text(
        0.05, 0.85,
        rf"$N={len(y_true)}$" + "\n"
        rf"$\sigma={sigma:.2f}$" + "\n"
        rf"$NMAD={sigma_nmad:.2f}$" + "\n"
        rf"$\eta={eta:.2f}\%$",
        transform=ax1.transAxes,
        fontsize=10,
        verticalalignment="top"
    )

    ax2.scatter(y_true, delta_z, s=5, c="black", alpha=0.7)
    ax2.axhline(0, color="red", linestyle="--", linewidth=1)
    ax2.axhline(0.15, color="blue", linestyle="--", linewidth=1)
    ax2.axhline(-0.15, color="blue", linestyle="--", linewidth=1)

    ax2.set_xlim(0, zmax)
    ax2.set_xlabel(r"$z_{\rm spec}$")
    ax2.set_ylabel(r"$\frac{z_{\rm spec}-z_{\rm photo}}{1+z_{\rm spec}}$")

    plt.tight_layout()
    plt.show()

Original ELAIS-S1 dataset shape: (495, 122)
Original CDFS dataset shape: (816, 122)
Best parameters: {'rf__max_depth': 20, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 2, 'rf__n_estimators': 300}
Best CV eta_0.15: 0.13726530612244897
eta_0.15: 14.95%
eta_2sigma: 6.37%
sigma: 0.1578
sigma_NMAD: 0.0664
R²: 0.4364
MSE: 0.0917
RMSE: 0.3028
MAE: 0.1462
